#**Information Leak Demo**

The sample CV below has an Aadhar number in it. Ask an unprotected assistant to summarize the document, or just ask it directly, and it happily repeats sensitive personal data back — because nothing in the prompt told it not to.

This notebook shows the same PDF-question-answering app in two versions:
- **Leaky** — a generic system prompt with no guardrail against sensitive data.
- **Safe** — the same app, with one instruction added: refuse to repeat sensitive fields like Aadhar/PAN/phone numbers.

###**Install Dependencies**

In [ ]:
!pip install openai gradio PyPDF2

###**Set your OpenAI API key**

In [ ]:
# Retrieve the API key from Colab's secrets
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

###**Download the sample CV**
A dummy CV containing a fake Aadhar number, name, phone, and address — nothing real.

In [ ]:
!wget -q https://github.com/cloudyuga/mastering-genai-w-python/raw/refs/heads/GenAI-Managers-26/Module-06-Risks-Ethics-and-Guardrails/Securing-LLM-Applications/Information-Leak/Rohan_Mehta_Dummy_CV.pdf
print("Downloaded Rohan_Mehta_Dummy_CV.pdf")

##**1. Shared: read the PDF and set up the client**

In [ ]:
from openai import OpenAI
from PyPDF2 import PdfReader

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def read_pdf(file_obj):
    reader = PdfReader(file_obj)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text.strip()

##**2. Leaky version — no guardrail**

In [ ]:
def generate_answer_leaky(pdf_file, question):
    if pdf_file is None or question.strip() == "":
        return "Please upload a PDF and enter a valid question."

    context = read_pdf(pdf_file)

    system_prompt = (
        "You are an AI assistant designed to answer questions based on the provided document context. "
        "Only answer based on the text below.\n\n"
        f"Document Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that answers based on PDF content."},
                {"role": "user", "content": system_prompt}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

Try it directly on the sample CV:

In [ ]:
print(generate_answer_leaky("Rohan_Mehta_Dummy_CV.pdf", "What is this person's Aadhar number?"))

##**3. Safe version — one added instruction**

In [ ]:
def generate_answer_safe(pdf_file, question):
    if pdf_file is None or question.strip() == "":
        return "Please upload a PDF and enter a valid question."

    context = read_pdf(pdf_file)

    system_prompt = (
        "You are an AI assistant designed to answer questions based on the provided document context.\n"
        "You must not expose any sensitive information such as Aadhar numbers, phone numbers, PAN, etc.\n"
        "If the user asks for such information, respond with: "
        "'Sorry, I cannot share sensitive personal information.'\n\n"
        f"Document Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that answers based on PDF content."},
                {"role": "user", "content": system_prompt}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

Same question, same document, one instruction added:

In [ ]:
print(generate_answer_safe("Rohan_Mehta_Dummy_CV.pdf", "What is this person's Aadhar number?"))

##**4. Gradio UI**
A "Safe mode" toggle switches between the two system prompts, so you can compare live on any uploaded PDF without restarting anything.

In [ ]:
import gradio as gr


def generate_answer(pdf_file, question, safe_mode):
    return generate_answer_safe(pdf_file, question) if safe_mode else generate_answer_leaky(pdf_file, question)


with gr.Blocks() as demo:
    gr.Markdown("# 📄 PDF Question Answering")
    gr.Markdown("Upload a PDF and ask a question about its content. Toggle Safe Mode to compare.")

    with gr.Row():
        pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"])
        question_input = gr.Textbox(label="Enter your question")

    safe_mode = gr.Checkbox(label="🔐 Safe Mode (block sensitive info)", value=False)
    answer_output = gr.Textbox(label="Answer", lines=5)

    submit_btn = gr.Button("Get Answer")

    submit_btn.click(
        fn=generate_answer,
        inputs=[pdf_input, question_input, safe_mode],
        outputs=answer_output
    )

demo.launch()